## Retrieval-Augmented Generation (RAG) Pipeline Explanation

This notebook demonstrates a complete RAG pipeline, which combines the strengths of large language models (LLMs) with external knowledge retrieval. The goal is to enable LLMs to generate more accurate, up-to-date, and contextually relevant responses by first retrieving relevant information from a knowledge base.

The pipeline can be broken down into two main phases:
1.  **Ingestion Phase**: Processing and storing external knowledge into a searchable format.
2.  **Retrieval and Generation Phase**: Using a user query to retrieve relevant information and then feeding that information to an LLM to generate a response.

### Overall Pipeline Flow:

1.  **Data Loading**: Load raw data (e.g., text files, PDFs) into the system.
2.  **Document Chunking**: Split large documents into smaller, manageable chunks.
3.  **Embedding Generation**: Convert these text chunks into numerical vector representations (embeddings).
4.  **Vector Store**: Store these embeddings in a vector database for efficient semantic search.
5.  **Query Embedding**: Convert a user's query into an embedding.
6.  **Retrieval**: Use the query embedding to find the most semantically similar document chunks from the vector store.
7.  **Context Augmentation**: Pass the retrieved chunks along with the original query to an LLM.
8.  **Response Generation**: The LLM uses this augmented context to generate a more informed and accurate answer.

### 1. Setup and Installations

Before starting, necessary libraries for LangChain, PDF processing, embedding models, and vector databases are installed.

In [1]:
!pip install langchain langchain-core langchain-community pypdf pymupdf sentence-transformers chromadb

In [2]:
from langchain_core.documents import Document

In [3]:
!pip install langchain-community

In [4]:
# Text data
from langchain_community.document_loaders.text import TextLoader
loader = TextLoader("/content/drive/MyDrive/RAG/data/Python.txt", encoding="utf-8")

/tmp/ipykernel_7033/2677991682.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders.text import TextLoader


Loads a text file (`Python.txt`) using `TextLoader` from `langchain_community.document_loaders.text`. This creates a `Document` object.

In [5]:
document = loader.load()

Executes the loading operation for the text file.

In [6]:
document

[Document(metadata={'source': '/content/drive/MyDrive/RAG/data/Python.txt'}, page_content='Python is a high-level, interpreted programming language that has become one of the most popular and widely used languages in the world. Created by Guido van Rossum and first released in 1991, Python emphasizes simplicity and readability, making it easy for beginners to learn while remaining powerful for experienced developers. Its clean and concise syntax allows programmers to write fewer lines of code compared to many other languages, enhancing productivity and maintainability. Python supports multiple programming paradigms, including procedural, object-oriented, and functional programming, which makes it versatile for a wide range of applications.\nSome key features and benefits of Python include:\n* Ease of Learning: Simple syntax and readability make Python beginner-friendly.\n* Versatility: Suitable for web development, data analysis, artificial intelligence, machine learning, scientific co

Displays the loaded `Document` object, showing its content and metadata.

INGESTION PIPELINE

In [7]:
# Data => Documents
import os
from langchain_community.document_loaders.pdf import PyPDFLoader

Imports `os` for directory operations and `PyPDFLoader` from `langchain_community.document_loaders.pdf` to handle PDF files.

DOCUMENTS

In [8]:
def load_all_pdfs():
    folder_path = "/content/drive/MyDrive/RAG/data/pdfs"
    num_docs = 0
    all_docs = []

    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".pdf"):
            # complete file path
            pdf_path = os.path.join(folder_path, filename)

            loader = PyPDFLoader(pdf_path)
            doc = loader.load()

            all_docs.extend(doc)
            num_docs += 1

    print("total pdfs:", num_docs)
    print("total pages:", len(all_docs))
    return all_docs

Defines the `load_all_pdfs` function. This function iterates through a specified directory (`/content/drive/MyDrive/RAG/data/pdfs`), identifies PDF files, and uses `PyPDFLoader` to load each PDF. It then aggregates all pages from all PDFs into a single list of `Document` objects.

In [9]:
!pip install langchain langchain-core langchain-community pypdf pymupdf sentence-transformers chromadb

In [10]:
all_pdf_documents = load_all_pdfs()

total pdfs: 2
total pages: 32


Calls `load_all_pdfs()` to execute the PDF loading process and stores the results in `all_pdf_documents`.

CHUNKS

In [11]:
!pip install langchain_text_splitters

In [12]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_docs(documents, chunk_size=500, chunk_overlap=50):

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap
    )

    chunked_docs = text_splitter.split_documents(documents)
    return chunked_docs

Imports `RecursiveCharacterTextSplitter` from `langchain_text_splitters` and defines `split_docs` function. This function takes a list of documents, `chunk_size`, and `chunk_overlap` to intelligently split the text while trying to keep related content together.

In [13]:
chunks = split_docs(all_pdf_documents)

Calls `split_docs` with `all_pdf_documents` to generate the chunks.

In [14]:
len(chunks)

321

Outputs the total number of generated chunks.

### 3. Document Chunking

Large documents are split into smaller, overlapping chunks to ensure that each piece of information is contained within a reasonable size for the LLM, and to capture contextual continuity across chunks.

---

EMBEDDINGS

In [15]:
from sentence_transformers import SentenceTransformer

Imports `SentenceTransformer` from the `sentence_transformers` library.

In [16]:
class EmbeddingManager:
    def __init__(self, model_name="all-MiniLM-L6-v2"):

        self.model_name=model_name
        print("loading model....", self.model_name)
        self.model = SentenceTransformer(self.model_name)
        print("embedding dimensions=", self.model.get_sentence_embedding_dimension())


    def generate_embeddings(self, text):
        embeddings = self.model.encode(text, show_progress_bar=True)
        print("embeddings shape:", embeddings.shape)
        return embeddings

Defines the `EmbeddingManager` class. This class initializes a `SentenceTransformer` model (e.g., `all-MiniLM-L6-v2`) and provides a `generate_embeddings` method to convert text into embeddings.

In [17]:
embedding_manager = EmbeddingManager()

loading model.... all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

embedding dimensions= 384


/tmp/ipykernel_7033/4021223045.py:7: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("embedding dimensions=", self.model.get_sentence_embedding_dimension())


Instantiates `EmbeddingManager`, loading the pre-trained sentence transformer model.

### 4. Embedding Generation

Each text chunk is converted into a high-dimensional numerical vector (embedding). These embeddings capture the semantic meaning of the text, allowing for similarity comparisons.

---

VECTOR STORE

In [18]:
import chromadb
import uuid

Imports `chromadb` for the vector database and `uuid` for generating unique IDs for documents.

In [19]:
class VectorStoreManager:
    def __init__(self, persist_directory="data/vector_store", collection_name="pdf_documents"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.collection = None
        self.client = None

        self._initialize_store()

    def _initialize_store(self):
        os.makedirs(self.persist_directory, exist_ok=True)

        # create a client
        self.client = chromadb.PersistentClient(path=self.persist_directory)

        # create the collection
        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={"description": "vector store collection for pdf embeddings in RAG"}
        )

        print("initialized the vector store with collection:", self.collection_name)
        print("docs in collection:", self.collection.count())

    def add_documents(self, documents, embeddings):
        if len(documents) != len(embeddings):
            raise ValueError("num of documents does not match num of embeddings")


        # store => ids, embedding, document, metadata
        ids = []
        all_metadata = []
        documents_content = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4()}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            all_metadata.append(metadata)

            documents_content.append(doc.page_content)

            embeddings_list.append(embedding.tolist())

            self.collection.add(
                ids=ids,
                metadatas=all_metadata,
                documents=documents_content,
                embeddings=embeddings_list
            )

        print("total documents added in vector store=", len(documents_content))
        print("docs in collection:", self.collection.count())

Defines the `VectorStoreManager` class. This class handles the initialization of a persistent ChromaDB client and collection. It also includes an `add_documents` method to store document chunks, their metadata, and embeddings into the collection.

In [20]:
vector_store = VectorStoreManager()

initialized the vector store with collection: pdf_documents
docs in collection: 321


Instantiates `VectorStoreManager`, which initializes the ChromaDB client and collection.

In [21]:
# data => documents => chunks => embeddings => store in vector store

texts = [doc.page_content for doc in chunks]

emebedding = embedding_manager.generate_embeddings(texts)

vector_store.add_documents(chunks, emebedding)

Batches:   0%|          | 0/11 [00:00<?, ?it/s]

embeddings shape: (321, 384)
total documents added in vector store= 321
docs in collection: 642


This cell orchestrates the final part of the ingestion phase: it extracts page content from the `chunks`, generates embeddings using the `embedding_manager`, and then adds these chunks and embeddings to the `vector_store`.

### 5. Vector Store Management (ChromaDB)

The generated embeddings, along with their corresponding text chunks and metadata, are stored in a vector database. ChromaDB is used here, providing persistence and efficient retrieval capabilities.

---

RETRIEVAL PIPELINE

In [22]:
from sklearn.metrics.pairwise import cosine_similarity

Imports `cosine_similarity` for calculating similarity between embeddings.

In [23]:
class RAGRetriever:
    def __init__(self, embedding_manager, vector_store):
        self.embedding_manager = embedding_manager
        self.vector_store = vector_store


    def retrieve(self, query, top_k=5, score_threshold=0.0):
        # query => embedding
        query_embeddings = self.embedding_manager.generate_embeddings([query])[0]

        # semantic search
        results = self.vector_store.collection.query(
            query_embeddings=[query_embeddings.tolist()],
            n_results=top_k
        )

        # cosine similarity
        retrieved_docs=[]

        if results["documents"] and results["documents"][0]:
            ids = results["ids"][0]
            metadatas = results["metadatas"][0]
            documents = results["documents"][0]
            distances = results["distances"][0]

            for i, (doc_id, metadata, document, distance) in enumerate(zip(ids, metadatas, documents, distances)):
                similarity_score = 1 - distance

                if similarity_score >= score_threshold:
                    retrieved_docs.append({
                        "id": doc_id,
                        "document": document,
                        "metadata": metadata,
                        "distance": distance,
                        "similarity_score": similarity_score,
                        "rank" : i + 1
                    })

            print(f"retrieved {len(retrieved_docs)} documents")

        else:
            print("no documents found")

        return retrieved_docs

Defines the `RAGRetriever` class. This class takes the `embedding_manager` and `vector_store` to perform retrieval. Its `retrieve` method generates an embedding for the input `query`, queries the `ChromaDB` collection for similar documents, calculates similarity scores, and returns the top `k` relevant documents.

In [24]:
rag_retriever = RAGRetriever(embedding_manager, vector_store)

Instantiates `RAGRetriever` with the previously created `embedding_manager` and `vector_store`.

In [25]:
rag_retriever.retrieve("What is encoder decoder")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape: (1, 384)
retrieved 5 documents


[{'id': 'doc_8b20f75f-c6bd-49f0-946e-73bf43e2a537',
  'document': 'positional encodings in both the encoder and decoder stacks. For the base model, we use a rate of\nPdrop = 0.1.\n7\naniketpatel161161@gmail.com',
  'metadata': {'language': 'en-US',
   'producer': 'pdfcpu v0.15.0 dev',
   'subject': 'Neural Information Processing Systems http://nips.cc/',
   'creationdate': '2026-10-01T08:05:56+00:00',
   'total_pages': 11,
   'author': 'Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser, Illia Polosukhin',
   'lastpage': '6008',
   'source': '/content/drive/MyDrive/RAG/data/pdfs/research1.pdf',
   'doc_index': 294,
   'publisher': 'Curran Associates, Inc.',
   'content_length': 140,
   'description-abstract': 'The dominant sequence transduction models are based on complex recurrent orconvolutional neural networks in an encoder and decoder configuration. The best performing such models also connect the encoder and decoder through an at

Demonstrates the retrieval process by calling `rag_retriever.retrieve` with a sample query, showing the retrieved documents.

### 6. Retrieval

Given a user query, this step involves converting the query into an embedding and performing a semantic search within the vector store to find the most relevant document chunks.

---

Integrate with LLMs

OPENAI - GPT

In [ ]:
API_KEY_OPENAI ="PASTE THE KEY FOR OPENAI HERE .... IT IS A SECRET KEY, DO NOT SHARE IT WITH ANYONE. KEEP IT SAFE AND SECURE. "

Stores the OpenAI API key. **Note**: For production use, it's recommended to secure API keys using environment variables or secret management services rather than hardcoding them.

In [27]:
!pip install langchain-openai

### 2. Data Ingestion (Loading Documents)

This step involves loading external data, which can be in various formats like text files or PDFs, into a structured format that LangChain can process (e.g., `Document` objects). The notebook demonstrates loading both a simple text file and multiple PDF files from a specified directory.

---

In [28]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    openai_api_key=API_KEY_OPENAI,
    model="gpt-5.4",
    temperature=0.1,
    max_tokens=1024
)

Initializes the `ChatOpenAI` LLM from `langchain_openai`. It's configured with the API key, a specific model (`gpt-5.4`), `temperature` for creativity, and `max_tokens` for response length.

In [29]:
# generate our retrieval-augmented output
def generate_output(query, retriever, llm, top_k=3):
    results = retriever.retrieve(query, top_k)

    context = "\n".join([doc["document"] for doc in results]) if results else ""

    if not context:
        print("we found no relevant context for the given query")

    # context + query
    prompt = f""" use given context to generate the answer for the query
                Context: {context}
                Query: {query} """

    response = llm.invoke(prompt) # expecting a string as prompt
    return response.content

Defines the `generate_output` function, which is the core of the RAG system's response generation. It takes a `query`, the `retriever`, and the `llm`.
    1.  It first uses the `retriever` to get relevant documents based on the `query`.
    2.  It constructs a `context` string from the retrieved documents.
    3.  It then creates a `prompt` that combines the `context` and the original `query`.
    4.  Finally, it invokes the `llm` with this augmented `prompt` to get the answer.

In [33]:
answer = generate_output("what is RAG?", rag_retriever, llm)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape: (1, 384)
retrieved 3 documents


OpenAIRateLimitError: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}

Calls `generate_output` with a sample query ("what is RAG?") to get an answer, demonstrating the end-to-end RAG process.

### 7. LLM Integration and Response Generation

Finally, the retrieved context is used to augment the user's query, which is then sent to a Large Language Model (LLM) to generate a coherent and informed answer.

---

We need to add credits to our OpenAI API account to use its AI models and features in our project. OpenAI charges us based on how much we use the API, such as the number of input and output tokens processed. These credits help us access the required features until the available balance is exhausted.